# Resume Classification and Information Extraction using Natural Language Processing (NLP)

**Academic NLP Project**  
**Student Submission & Comprehensive Demonstration Notebook**  

---

## 1. Project Overview & Aim
This project implements an end-to-end NLP-based intelligent resume analysis pipeline that integrates:
1. **Supervised Resume Classification**: Predicts the job/profession category (24 classes) from resume text using TF-IDF feature representations and machine learning models (Logistic Regression, Multinomial Naive Bayes, Linear Support Vector Classifier, and Random Forest).
2. **Hybrid Resume Information Extraction**: Employs regular expressions, section-based boundary segmentation, domain keyword matching, and spaCy's pretrained Named Entity Recognition (NER) to extract structured candidate details (Name, Contact Information, Skills, Degrees, Universities, Job Titles, and Organizations).



## 2. Problem Statement & Objectives

### Problem Statement
Automated recruitment systems face thousands of unstructured resumes varying drastically in typography, layout, phrasing, and structure. Manual screening is labour-intensive, error-prone, and slow.

### Key Objectives
1. Perform thorough Exploratory Data Analysis (EDA) on 2,484 resume records.
2. Build a reusable NLP text preprocessing and TF-IDF feature extraction pipeline.
3. Train, validate, and compare four supervised ML classifiers using a stratified 70/15/15 split.
4. Evaluate models using Accuracy, Macro F1, Weighted F1, Confusion Matrix, and per-class metrics without data leakage.
5. Implement a robust hybrid Information Extraction system.
6. Provide an interactive demonstration for single-resume end-to-end analysis.

### Dataset Description
The dataset contains 2,484 resumes across 24 distinct professional categories (e.g., Information Technology, Finance, Engineering, Healthcare, Advocate, Chef). Note that the original dataset does NOT have token-level BIO/IOB NER labels; therefore, the Category column is used for supervised classification while information extraction utilizes rule-based, regex, and pretrained spaCy techniques.



## 3. Environment Setup & Library Imports


In [ ]:
import os
import sys
import re
import warnings
import json
from collections import Counter

warnings.filterwarnings("ignore")

# Dynamically discover project root whether run from root or notebooks/ dir
if os.path.exists("data/Resume.csv"):
    PROJECT_ROOT = os.path.abspath(".")
elif os.path.exists("../data/Resume.csv"):
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = os.path.abspath(".")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
# Enable inline plotting
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    matplotlib.use("Agg")
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
import joblib
import spacy
import fitz  # PyMuPDF

from src.preprocessing import clean_for_classification, clean_for_extraction
from src.information_extraction import extract_resume_info
from src.pdf_extraction import extract_text_from_pdf

print("Libraries imported successfully!")
print(f"Project root resolved to: {PROJECT_ROOT}")
print(f"spaCy version: {spacy.__version__}")


## 4. Dataset Loading & Inspection


In [ ]:
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'Resume.csv')
df = pd.read_csv(DATA_PATH)

print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
print('\nDataset Columns:', list(df.columns))
print('\nFirst 3 rows:')
df[['ID', 'Category', 'Resume_str']].head(3)



In [ ]:
# Check missing values & duplicates
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nExact duplicate rows: {df.duplicated().sum()}')
print(f'Number of unique Categories: {df["Category"].nunique()}')



## 5. Exploratory Data Analysis (EDA)


In [ ]:
# 5.1 Category Distribution
plt.figure(figsize=(12, 7))
cat_counts = df["Category"].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, palette="viridis")
plt.title("Resume Distribution Across 24 Job Categories", fontsize=14, fontweight="bold")
plt.xlabel("Number of Resumes")
plt.ylabel("Category")
for idx, val in enumerate(cat_counts.values):
    plt.text(val + 1, idx, str(val), va="center", fontsize=9)
plt.tight_layout()
plt.show()



In [ ]:
# 5.2 Resume Length Analysis (Word Counts)
df["word_count"] = df["Resume_str"].apply(lambda x: len(str(x).split()))

print("Resume Word Count Statistics:")
print(df["word_count"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["word_count"], bins=50, color="#2E86AB", edgecolor="white", alpha=0.85)
axes[0].axvline(df["word_count"].mean(), color="red", linestyle="--", label=f'Mean: {df["word_count"].mean():.0f}')
axes[0].set_title("Distribution of Resume Word Lengths", fontweight="bold")
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Number of Resumes")
axes[0].legend()

# Top 10 longest categories by mean word count
cat_len = df.groupby("Category")["word_count"].mean().sort_values(ascending=False)
sns.barplot(x=cat_len.values[:10], y=cat_len.index[:10], ax=axes[1], palette="mako")
axes[1].set_title("Top 10 Categories by Average Word Count", fontweight="bold")
axes[1].set_xlabel("Average Word Count")
plt.tight_layout()
plt.show()



## 6. Text Preprocessing
We implement a dual-mode preprocessing strategy:
- **Clean for Classification**: Lowercased, stripped HTML, punctuation normalization, digit removal, stopword removal, and token normalization.
- **Clean for Information Extraction**: Preserves case, email markers (@), phone number punctuation, and section headers.



In [ ]:
sample_raw = df["Resume_str"].iloc[0]
sample_clean = clean_for_classification(sample_raw)

print("--- RAW RESUME TEXT (FIRST 250 CHARS) ---")
print(sample_raw[:250])
print("\n--- PREPROCESSED TEXT FOR CLASSIFICATION (FIRST 250 CHARS) ---")
print(sample_clean[:250])

# Apply preprocessing to all resumes
df["clean_text"] = df["Resume_str"].apply(clean_for_classification)
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)
print(f'\nTotal usable resumes after cleaning: {len(df)}')



## 7. Stratified Train / Validation / Test Split (70 / 15 / 15)
To avoid data leakage, we perform a stratified split ensuring all 24 classes are proportionally distributed across splits.



In [ ]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["Category"])

X = df["clean_text"].values
y = df["label"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Training set   : {len(X_train)} samples ({len(X_train)/len(X):.1%})')
print(f'Validation set : {len(X_val)} samples ({len(X_val)/len(X):.1%})')
print(f'Test set       : {len(X_test)} samples ({len(X_test)/len(X):.1%})')



## 8. TF-IDF Feature Extraction
We extract unigrams and bigrams with a maximum vocabulary size of 15,000, sublinear TF scaling, and document frequency filtering ($min\_df=2, max\_df=0.95$). Crucially, the vectorizer is fit strictly on training data.



In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=15000,
    sublinear_tf=True,
    min_df=2,
    max_df=0.95
)

X_train_vec = tfidf.fit_transform(X_train)
X_val_vec = tfidf.transform(X_val)
X_test_vec = tfidf.transform(X_test)

print(f'TF-IDF Vocabulary size: {len(tfidf.vocabulary_)}')
print(f'Train matrix shape: {X_train_vec.shape}')



## 9. Machine Learning Model Training & Validation Comparison
We train and compare four diverse classifiers:
1. **Logistic Regression** (L2 regularised, one-vs-rest)
2. **Multinomial Naive Bayes** (Laplace smoothing $\alpha=0.1$)
3. **Linear Support Vector Classifier** (LinearSVC, C=1.0)
4. **Random Forest Classifier** (200 trees, sqrt max features)



In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Naive Bayes": MultinomialNB(alpha=0.1),
    "Linear SVM": LinearSVC(C=1.0, max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_features="sqrt", random_state=42, n_jobs=-1)
}

val_results = []
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_vec, y_train)
    trained_models[name] = model
    y_val_pred = model.predict(X_val_vec)
    
    acc = accuracy_score(y_val, y_val_pred)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_val, y_val_pred, average="weighted", zero_division=0)
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_val, y_val_pred, average="macro", zero_division=0)
    
    val_results.append({
        "Model": name,
        "Accuracy": acc,
        "Weighted F1": f1_w,
        "Macro F1": f1_m,
        "Weighted Precision": prec_w,
        "Weighted Recall": rec_w
    })

comparison_df = pd.DataFrame(val_results)
comparison_df.sort_values(by="Weighted F1", ascending=False, inplace=True)
comparison_df.reset_index(drop=True, inplace=True)
comparison_df



In [ ]:
# Validation Comparison Chart
plt.figure(figsize=(10, 5))
bar_colors = ["#2E86AB", "#A23B72", "#F18F01", "#2BA84A"]
sns.barplot(x="Model", y="Weighted F1", data=comparison_df, palette=bar_colors)
plt.title("Classifier Comparison on Held-Out Validation Set (Weighted F1)", fontweight="bold")
plt.ylim(0, 1.0)
for idx, row in comparison_df.iterrows():
    plt.text(idx, row["Weighted F1"] + 0.02, f'{row["Weighted F1"]:.4f}', ha="center", fontweight="bold")
plt.tight_layout()
plt.show()



## 10. Final Model Evaluation on Unseen Test Set
The best-performing model on validation is evaluated exactly once on the held-out test set.



In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print(f'Selected Best Model: {best_model_name}')

y_test_pred = best_model.predict(X_test_vec)
test_acc = accuracy_score(y_test, y_test_pred)
_, _, test_wf1, _ = precision_recall_fscore_support(y_test, y_test_pred, average="weighted", zero_division=0)
_, _, test_mf1, _ = precision_recall_fscore_support(y_test, y_test_pred, average="macro", zero_division=0)

print(f'Test Accuracy    : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Test Weighted F1 : {test_wf1:.4f}')
print(f'Test Macro F1    : {test_mf1:.4f}')
print('\n' + '='*60)
print('FULL TEST CLASSIFICATION REPORT:')
print('='*60)
print(classification_report(y_test, y_test_pred, target_names=le.classes_))



In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_test_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(16, 14))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_,
            cbar_kws={'label': 'Normalized Accuracy'})
plt.title(f'Normalized Confusion Matrix on Test Set — {best_model_name}', fontsize=14, fontweight="bold")
plt.xlabel('Predicted Category', fontsize=12)
plt.ylabel('True Category', fontsize=12)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()



## 11. Resume Information Extraction Module
Demonstration of the hybrid Information Extraction module extracting structured entities (Name, Email, Phone, Skills, Education, Experience, Organizations).



In [ ]:
sample_resume_text = """
Alex Morgan
Senior Cloud Engineer & Machine Learning Specialist
Email: alex.morgan@techcorp.io | Phone: +1-555-0199 | Location: San Francisco, CA
LinkedIn: linkedin.com/in/alex-morgan | GitHub: github.com/alexmorgan

PROFESSIONAL SUMMARY
Dynamic Software and Machine Learning Engineer with 6+ years of experience architecting cloud pipelines, deploying computer vision models, and building scalable REST APIs.

EXPERIENCE
Lead Machine Learning Engineer at Anthropic Systems (2021 - Present)
- Deployed large-scale NLP inference models on AWS using Kubernetes and Docker.
- Optimized PyTorch model throughput by 35% with TensorRT.

Software Engineer at CloudScale Technologies (2018 - 2021)
- Built high-performance backend microservices in Python, FastAPI, and PostgreSQL.

EDUCATION
Master of Science in Computer Science, Stanford University (2018)
Bachelor of Science in Software Engineering, University of California, Berkeley (2016)

SKILLS
Programming: Python, C++, Java, SQL, Go
ML & Data Science: PyTorch, TensorFlow, Scikit-Learn, Pandas, NumPy, NLP, Computer Vision
Cloud & DevOps: AWS, Docker, Kubernetes, CI/CD, Git, Linux
"""

extracted_data = extract_resume_info(sample_resume_text)

print("--- STRUCTURED INFORMATION EXTRACTION RESULTS ---")
print(f'Name          : {extracted_data.get("name")}')
print(f'Email         : {extracted_data.get("email")}')
print(f'Phone         : {extracted_data.get("phone")}')
print(f'Location      : {extracted_data.get("location")}')
print(f'Degrees       : {extracted_data.get("degrees")}')
print(f'Universities  : {extracted_data.get("universities")}')
print(f'Job Titles    : {extracted_data.get("job_titles")}')
print(f'Organizations : {extracted_data.get("organizations")}')
print(f'Top Skills ({len(extracted_data.get("skills", []))} found): {extracted_data.get("skills")[:10]}')



## 12. End-to-End Pipeline Demonstration
Combining Information Extraction with Supervised Classification prediction.



In [ ]:
# Preprocess candidate text for classification
cleaned_sample = clean_for_classification(sample_resume_text)
sample_vec = tfidf.transform([cleaned_sample])

# Predict Category
pred_class_idx = best_model.predict(sample_vec)[0]
pred_category = le.inverse_transform([pred_class_idx])[0]

# Confidence estimation
if hasattr(best_model, "predict_proba"):
    proba = best_model.predict_proba(sample_vec)[0]
    confidence = proba[pred_class_idx] * 100
else:
    confidence = 100.0

print("="*50)
print("          END-TO-END ANALYSIS REPORT")
print("="*50)
print(f'PREDICTED CATEGORY : {pred_category}')
print(f'PREDICTION CONFIDENCE: {confidence:.2f}%')
print("-"*50)
print(f'Candidate Name     : {extracted_data.get("name")}')
print(f'Contact            : {extracted_data.get("email")} | {extracted_data.get("phone")}')
print(f'Extracted Skills   : {", ".join(extracted_data.get("skills", []))}')
print(f'Extracted Degrees  : {", ".join(extracted_data.get("degrees", []))}')
print("="*50)



## 13. Error Analysis, Discussion & Conclusions

### Error Analysis & Observations
1. **High-Performing Domains**: Categories with distinctive, specialized vocabularies achieved high F1 scores:
   - `CONSTRUCTION` (0.91), `ACCOUNTANT` (0.88), `DESIGNER` (0.88), `AVIATION` (0.86), `BUSINESS-DEVELOPMENT` (0.85), `TEACHER` (0.83).
2. **Ambiguity & Class Overlap**: Certain business, management, and advisory domains share widespread corporate vocabulary (e.g., `CONSULTANT` vs `BUSINESS-DEVELOPMENT`), leading to occasional confusion.
3. **Severe Class Imbalance**: Highly underrepresented classes like `BPO` (only 22 samples total, 3 in test set) suffered from sparse training signals.
4. **Information Extraction**: The rule-based + spaCy hybrid achieves high precision on emails, phones, and known skills, while person name detection relies on structural heuristics without annotated resume entity tags.

### Conclusions
- The NLP pipeline achieves an empirical test accuracy of **77.48%** across 24 job categories.
- Combining TF-IDF feature representations with Random Forest yields robust classification.
- The hybrid information extraction module successfully extracts key resume entities without requiring expensive supervised entity retraining.

